In [ ]:
!pip install -U peft bitsandbytes transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 123.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 11.8 MB/s eta 0:00:00


In [ ]:
from datasets import Dataset, load_dataset
dataset = load_dataset("JAYASREESS/semantic_integrity", split="train")

In [ ]:
!pip install PyMuPDF

In [ ]:
print(dataset)

Dataset({
    features: ['pdf'],
    num_rows: 4
})


In [ ]:
# Removed: This installation caused a conflict with PyMuPDF
# pip install fitz

In [ ]:
!pip install pdfplumber

In [ ]:
!pip uninstall -y fitz PyMuPDF
!pip install -U PyMuPDF

Found existing installation: fitz 0.0.1.dev2
Uninstalling fitz-0.0.1.dev2:
  Successfully uninstalled fitz-0.0.1.dev2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 60.2 MB/s eta 0:00:00


In [ ]:
import fitz
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [ ]:
pdf_files = [
    "/content/dox1.pdf",
    "/content/dox 2-1.pdf",
    "/content/dox 3-1.pdf",
    "/content/dox 4-1.pdf"
]

pdf_texts = []
for pdf_file in pdf_files:
    extracted_blocks = extract_text_from_pdf(pdf_file)
    pdf_texts.extend(extracted_blocks)

In [ ]:


pdf_texts



["Clean Sale Deed \n \n09.02.2026 (Two thousand twenty-sixth year, February month, ninth day) Krishnagiri \nDistrict, Thenkanikottai Taluk. Thottathimmanahalli Post, Veppalampatti village, residing \nat door number 3/69, Late Govindan's son Sakthivel  to you. \n \nKrishnagiri District, Thenkanikottai Taluk, Thottathimmanahalli Post, Veppalampatti \nvillage, residing at door number 3/65, Chinnakannu's wife Rathinammal.1 , the above-\nmentioned 1st party Rathinammal & Chinnakannu's daughter, and Saakkappan's wife \nSanthi. \n. \nThe \nabove-mentioned \nKrishnagiri \nDistrict, \nThenkanikottai \nTaluk, \nThottathimmanahalli Post, Veppalampatti village, residing at door number 3/65, \nNakappan (alias Chinnasaakkappan)'s son Kannan (alias Kannappan). 3 , the above-\nmentioned Krishnagiri District, Thenkanikottai Taluk, Thottathimmanahalli Post, \nVeppalampatti village, residing at door number 3/65, Kannan (alias Kannappan)'s son \nLate Govindan's wife Roja.4  , the above-mentioned 4th party

In [ ]:
import re
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        # Split on double line breaks or long newlines
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:  # ignore too short lines
                paragraphs.append(clean)
    return paragraphs


In [ ]:
paragraphs = split_paragraphs(pdf_texts)

In [ ]:
data = [{"text": p} for p in paragraphs]

In [ ]:
print(data)


[{'text': "09.02.2026 (Two thousand twenty-sixth year, February month, ninth day) Krishnagiri \nDistrict, Thenkanikottai Taluk. Thottathimmanahalli Post, Veppalampatti village, residing \nat door number 3/69, Late Govindan's son Sakthivel  to you."}, {'text': "Krishnagiri District, Thenkanikottai Taluk, Thottathimmanahalli Post, Veppalampatti \nvillage, residing at door number 3/65, Chinnakannu's wife Rathinammal.1 , the above-\nmentioned 1st party Rathinammal & Chinnakannu's daughter, and Saakkappan's wife \nSanthi. \n. \nThe \nabove-mentioned \nKrishnagiri \nDistrict, \nThenkanikottai \nTaluk, \nThottathimmanahalli Post, Veppalampatti village, residing at door number 3/65, \nNakappan (alias Chinnasaakkappan)'s son Kannan (alias Kannappan). 3 , the above-\nmentioned Krishnagiri District, Thenkanikottai Taluk, Thottathimmanahalli Post, \nVeppalampatti village, residing at door number 3/65, Kannan (alias Kannappan)'s son \nLate Govindan's wife Roja.4  , the above-mentioned 4th party Roj

In [ ]:
from datasets import Dataset
dataset = Dataset.from_list(data)

In [ ]:
dataset

Dataset({
    features: ['text'],
    num_rows: 28
})

In [ ]:


# model_name = "meta-llama/Llama-2-7b-hf"
# model_name = "meta-llama/Llama-3.1-8B"
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling


In [ ]:


tokenizer = AutoTokenizer.from_pretrained(model_name)



In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:


def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens



In [ ]:


tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])



Map:   0%|          | 0/28 [00:00<?, ? examples/s]

In [ ]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 28
})

In [ ]:
from transformers import BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./llama-legal-domain",
    num_train_epochs=2,
    per_device_train_batch_size=1, # Reduced batch size
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

In [ ]:


from transformers import TrainingArguments
help(TrainingArguments)



Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

q_lora_model= get_peft_model(model, lora_config)

trainer = Trainer(
    model=q_lora_model,
    args=training_args,
    train_dataset=tokenized
)

In [ ]:
trainer.train()
q_lora_model.save_pretrained(training_args.output_dir)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
50,9.188898


In [ ]:

import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [ ]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

In [ ]:


tokenized



Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 28
})

In [ ]:
pip

In [ ]:
from transformers import BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)



In [ ]:
q_lora_model= get_peft_model(model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:


trainer = Trainer(
    model=q_lora_model,
    args=args,
    train_dataset=tokenized
)



In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
20,7.723189


TrainOutput(global_step=20, training_loss=7.723188781738282, metrics={'train_runtime': 81.4286, 'train_samples_per_second': 1.719, 'train_steps_per_second': 0.246, 'total_flos': 445407528222720.0, 'train_loss': 7.723188781738282, 'epoch': 5.0})

In [ ]:
model="/content/llama-legal-domain"

In [ ]:
# First, load the base model again
base_model = AutoModelForCausalLM.from_pretrained(
    model_name, # Use the original model name
    quantization_config=quantization_config,
    device_map="auto"
)

# Load the LoRA adapters from the saved directory
from peft import PeftModel
lora_model = PeftModel.from_pretrained(base_model, model)

# Merge the LoRA adapters with the base model
merged_model = lora_model.merge_and_unload()

# Now, `merged_model` is your fully fine-tuned model
# You can save it or use it for inference
print("LoRA adapters merged successfully into the base model.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:110: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


LoRA adapters merged successfully into the base model.


In [ ]:

prompt = "Clean deed documents for finding duplicates,inconsistencies and contradcitions"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = merged_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clean deed documents for finding duplicates,inconsistencies and contradcitions.
I think that the best way to resolve this situation is to go back to the very beginning of the history of the United States, to 1776.
It was then that the first treaty between the United States and Great Britain was signed in Paris, which provided a legal basis for the "division" of North America.
Since this document was not ratified by the United States Congress until April 25, 1789, and since it had


In [ ]:
import shutil
from google.colab import files

# Zip the directory containing the saved model
shutil.make_archive("tinyllama-lora", "zip", "./tinyllama-lora")

# Now download the zipped file
files.download("tinyllama-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
instruction fine tuning

In [ ]:


from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset



In [ ]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:


tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token



In [ ]:

import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-lora.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-20"

In [ ]:
non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
prompt = "Clean deed documents for finding duplicates,inconsistencies and contradcitions"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:


outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)



In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clean deed documents for finding duplicates,inconsistencies and contradcitions,and it can be easily used as a search engine to look up for a specific document or a set of documents.
The system is designed to work with the most popular tools, such as Windows Explorer and Microsoft Word.It also supports other common applications like Google Chrome and Mozilla Firefox.
The application also has an optional free plugin for Outlook 2013, allowing to add attachments to mail messages and copy them directly from the message into the document searcher.
An


In [ ]:
from datasets import Dataset
import pandas as pd

file_paths = [
    "/content/Contradiction.xlsx",
    "/content/Duplication.xlsx",
    "/content/Inconsistency.xlsx"
]

# Load each Excel file into a pandas DataFrame and store them in a list
dfs = []
for file_path in file_paths:
    df = pd.read_excel(file_path)
    dfs.append(df)

# Concatenate all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# Convert the pandas DataFrame to a datasets.Dataset object
dataset = Dataset.from_pandas(combined_df)
dataset

Dataset({
    features: ['SENTENCE 1', 'SENTENCE 2', 'LABEL'],
    num_rows: 344
})

In [ ]:
def format_example(example):
    prompt = f"### SENTENCE 1:\n{example['SENTENCE 1']}\n### SENTENCE 2:\n{example['SENTENCE 2']}\n### LABEL:\n{example['LABEL']}"
    return {"text": prompt}

In [ ]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/344 [00:00<?, ? examples/s]

In [ ]:
dataset


Dataset({
    features: ['SENTENCE 1', 'SENTENCE 2', 'LABEL', 'text'],
    num_rows: 344
})

In [ ]:


dataset['text'][0]



'### SENTENCE 1:\nThe land extent is 1.12 acres.\n### SENTENCE 2:\nThe land extent is 2.00 acres.\n### LABEL:\n0'

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:

def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens



In [ ]:


tokenized = dataset.map(tokenize_fn, batched=True)



Map:   0%|          | 0/344 [00:00<?, ? examples/s]

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
  lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
instruction_model = get_peft_model(non_instruction_model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=instruction_model,
    args=args,
    train_dataset=tokenized,
)

In [ ]:
trainer.train()

Step,Training Loss
20,6.298928
40,0.325896
60,0.225386
80,0.201747
100,0.185541
120,0.176187


TrainOutput(global_step=129, training_loss=1.1610703126404636, metrics={'train_runtime': 261.1913, 'train_samples_per_second': 3.951, 'train_steps_per_second': 0.494, 'total_flos': 3283289779470336.0, 'train_loss': 1.1610703126404636, 'epoch': 3.0})

In [ ]:


model_path = "/content/tinyllama-instruction"

In [ ]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:


prompt = "Find duplication in the document."



In [ ]:

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")


In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:


print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))




Model Output:

Find duplication in the document.

  * To search for duplicate text, in the **Search** pane, select **Text Search** from the **Search** group to open the **Text Search** dialog box. For more information about using this dialog box, see [Text Search](../../vsto/text-search.md).

  * To search for a substring of a word, in the **Find** pane, type the string you want to find. If no matches are found, then type


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import torch
import gc # Import gc for memory management

# --- Data Preparation ---
dataset = load_dataset("csv", data_files="/content/train_sample_10rows.csv", split="train")

def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}
dataset = dataset.map(format_example)

# Tokenizer is already loaded (global `tokenizer`)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_and_mask(example):
    text = example["text"]
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=512)
    response_marker = "### Response:"
    response_start = text.find(response_marker)

    if response_start != -1:
        response_token_start = len(tokenizer(text[:response_start], add_special_tokens=False)["input_ids"])
    else:
        response_token_start = 0

    labels = enc["input_ids"].copy()
    labels[:response_token_start] = [-100] * response_token_start
    enc["labels"] = labels
    return enc

tokenized = dataset.map(tokenize_and_mask, batched=False)
print("Tokenization + masking done.")

# --- Instruction Fine-Tuning ---

# Clear GPU memory before applying new LoRA and training
# Delete previous models to free GPU memory
if 'model' in globals():
    del model
if 'q_lora_model' in globals():
    del q_lora_model
if 'merged_model' in globals():
    del merged_model
# It's also important to delete previous trainer and associated models if they are no longer needed
if 'trainer' in globals():
    del trainer

gc.collect()
torch.cuda.empty_cache()

# non_instruction_model (from cell g2NsFw5PmO-d) is the base for instruction tuning.
# It's a PeftModel loaded from /content/checkpoint-5.

# LoRA config for instruction tuning
lora_config_instruction = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

# Apply the instruction-tuning LoRA config to the non_instruction_model
instruction_training_model = get_peft_model(non_instruction_model, lora_config_instruction)

# Training arguments (using the 'args' variable from previous cell QVFY-sRRl0z5)
training_args_instruction = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none",
    gradient_checkpointing=True # Enable gradient checkpointing for memory efficiency
)

trainer_instruction = Trainer(
    model=instruction_training_model,
    args=training_args_instruction,
    train_dataset=tokenized,
)

# Train the model
trainer_instruction.train()

# Save the instruction-tuned model (LoRA adapters) and tokenizer
trainer_instruction.save_model("/content/tinyllama-instruction")
tokenizer.save_pretrained("/content/tinyllama-instruction")

# --- Cleanup after training and before inference ---
del trainer_instruction
del instruction_training_model # Delete the training model
gc.collect()
torch.cuda.empty_cache()

# --- Inference and Comparison ---

# non_instruction_model (the globally loaded one from /content/checkpoint-5) is ready for inference.
# We need to explicitly load the base model to apply the instruction-tuned adapters for inference.

# Load base model with quantization for inference
quantization_config_inference = BitsAndBytesConfig(load_in_8bit=True)
base_model_for_inference = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T", # Original base model name
    quantization_config=quantization_config_inference,
    device_map="auto"
)

# Load the instruction-tuned adapters onto the base model
instruction_tuned_model_for_inference = PeftModel.from_pretrained(
    base_model_for_inference,
    "/content/tinyllama-instruction" # Path to the saved instruction-tuned adapters
)

# Test generation with an instruction-tuned prompt
prompt_instruction_test = "### Instruction:\n Detect duplication of land area values in the document.\n### Input:\n\n### Response:\n"
inputs_instruction_test = tokenizer(prompt_instruction_test, return_tensors="pt").to("cuda")

outputs_instruction_test = instruction_tuned_model_for_inference.generate(
    **inputs_instruction_test,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)
print("\nModel Output (Instruction-tuned):\n")
print(tokenizer.decode(outputs_instruction_test[0], skip_special_tokens=True))

# Iterate through questions for comparison
questions = [
    "Detect duplication of land area values in the document.",
    "Identify inconsistencies in property market value information.",
    "Find duplication in the document."
]

for q in questions:
    print(f"Question: {q}")

    # --- Non-instruction model (document-finetuned) ---
    print("\n--- Non-instruction model (document-finetuned) ---")
    inputs_non_inst = tokenizer(q, return_tensors="pt").to("cuda")
    # Use the globally available `non_instruction_model` from cell g2NsFw5PmO-d
    outputs_non_inst = non_instruction_model.generate(**inputs_non_inst, max_new_tokens=80)
    print(tokenizer.decode(outputs_non_inst[0], skip_special_tokens=True))

    # --- Instruction-tuned model ---
    print("\n--- Instruction-tuned model ---")
    prompt_inst = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    inputs_inst = tokenizer(prompt_inst, return_tensors="pt").to("cuda")
    outputs_inst = instruction_tuned_model_for_inference.generate(**inputs_inst, max_new_tokens=100)
    print(tokenizer.decode(outputs_inst[0], skip_special_tokens=True))
    print("="*80, "\n")

# Final cleanup of inference models
del base_model_for_inference
del instruction_tuned_model_for_inference
# del non_instruction_model # Keep this if it's meant to persist globally, otherwise uncomment to delete.
gc.collect()
torch.cuda.empty_cache()

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenization + masking done.


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.



Model Output (Instruction-tuned):

### Instruction:
 Detect duplication of land area values in the document.
### Input:

### Response:
* If the document contains duplicate land area values, an error will be returned to the caller.
* Otherwise, return 0 (success).

## Examples ##
### Example 1 ###
@Test
void test_DuplicateLandArea() throws Exception {
    LandArea lA = new LandArea();
    String docText = "\"This is a doc file.\"";
    long expected = 2; //expected number of lines
   
Question: Detect duplication of land area values in the document.

--- Non-instruction model (document-finetuned) ---
Detect duplication of land area values in the document.

















































































--- Instruction-tuned model ---
### Instruction:
Detect duplication of land area values in the document.
### Input:

### Response:

### Expected Output:

### Test Cases:

### Test Case 1:

### Test Case 2:

### Test Case 3:

### Test Case 4:

### Test Cas

First, ensure you have extracted the `merged_tinyllama_instruction.zip` file into your VSCode project directory. Let's assume you've placed it in a folder named `merged_tinyllama_instruction`.

Then, you can use the following Python code to load the model and tokenizer, and perform inference:

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import os

# Define the path to your saved merged model directory
# This path should be relative to where you run your Python script in VSCode
local_model_path = "./merged_tinyllama_instruction"

print(f"Loading model and tokenizer from: {local_model_path}")

# Check if the directory exists
if not os.path.exists(local_model_path):
    raise FileNotFoundError(f"Model directory not found at: {local_model_path}. "
                            "Please ensure you have unzipped 'merged_tinyllama_instruction.zip' "
                            "and placed the folder in the correct location.")

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

# Load the fine-tuned model
# If your GPU has enough memory, you can try loading it to CUDA directly
# model = AutoModelForCausalLM.from_pretrained(local_model_path, torch_dtype=torch.float16).to("cuda")
model = AutoModelForCausalLM.from_pretrained(local_model_path)

# Ensure pad_token is set if not already in the tokenizer config (good practice)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Create a text generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float32, # Use torch.float16 if your GPU supports it and memory is a concern
    device=0 if torch.cuda.is_available() else -1 # Use GPU if available, else CPU
)

print("Model loaded and pipeline created successfully. Ready for inference!")

# Example inference with an instruction-tuned prompt
prompt = "### Instruction:\nIdentify inconsistencies in property market value information.\n### Input:\nProperty A: Value $500,000, Date 2023-01-01. Property A: Value $450,000, Date 2023-06-01.\n### Response:\n"

print(f"\nGenerating response for prompt:\n{prompt}")

outputs = llm_pipeline(
    prompt,
    max_new_tokens=100,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1
)

# Print the generated text
generated_text = outputs[0]["generated_text"]
print("\nGenerated Output:")
print(generated_text)

# You can integrate this into your application by calling the llm_pipeline with dynamic prompts.


In [ ]:
model="/content/llama-legal-domain"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from peft import PeftModel
import torch
import gc

# Ensure previous models are cleared from memory if they are no longer needed
if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer
if 'llm' in globals():
    del llm

gc.collect()
torch.cuda.empty_cache()

# Define the path to the saved LoRA adapters from the first fine-tuning step
LORA_ADAPTERS_PATH = "/content/llama-legal-domain"
ORIGINAL_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Load the tokenizer (from the original model)
tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the base model with quantization config
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

base_model_for_inference = AutoModelForCausalLM.from_pretrained(
    ORIGINAL_MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

# Load the LoRA adapters onto the base model
document_finetuned_lora_model = PeftModel.from_pretrained(
    base_model_for_inference,
    LORA_ADAPTERS_PATH
)

# Merge the LoRA adapters with the base model for inference
merged_document_finetuned_model = document_finetuned_lora_model.merge_and_unload()

# Create the text-generation pipeline
llm = pipeline("text-generation", model=merged_document_finetuned_model, tokenizer=tokenizer)

print("Document-finetuned model loaded and pipeline created successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:110: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


Document-finetuned model loaded and pipeline created successfully.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Define the path to your saved merged model directory in your VSCode project
# Make sure this path is correct relative to where you run your script
local_model_path = "./my_finetuned_model_vscode" # Adjust this if you unzipped to a different location

print(f"Loading model and tokenizer from: {local_model_path}")

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

# Load the fine-tuned model
# If your merged model was saved with a specific torch_dtype (e.g., float16), you might specify it here
model = AutoModelForCausalLM.from_pretrained(local_model_path)

# Ensure pad_token is set if not already in the tokenizer config (good practice)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Create a text generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float32, # Use torch.float16 if your GPU supports it and memory is a concern
    device=0 if torch.cuda.is_available() else -1 # Use GPU if available, else CPU
)

print("Model loaded and pipeline created successfully. Ready for inference!")

# Example inference with an instruction-tuned prompt
prompt = "### Instruction:\nIdentify inconsistencies in property market value information.\n### Input:\nProperty A: Value $500,000, Date 2023-01-01. Property A: Value $450,000, Date 2023-06-01.\n### Response:\n"

print(f"\nGenerating response for prompt:\n{prompt}")

outputs = llm_pipeline(
    prompt,
    max_new_tokens=100,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1
)

# Print the generated text
generated_text = outputs[0]["generated_text"]
print("\nGenerated Output:")
print(generated_text)

# You can integrate this into your application by calling the llm_pipeline with dynamic prompts.


Loading model and tokenizer from: ./my_finetuned_model_vscode


OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './my_finetuned_model_vscode'.

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Define paths
ORIGINAL_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
INSTRUCTION_ADAPTERS_PATH = "/content/tinyllama-instruction" # Path where instruction-tuned adapters were saved
OUTPUT_MERGED_MODEL_DIR = "/content/merged_tinyllama_instruction"

# Clear GPU memory (optional, but good practice)
gc.collect()
torch.cuda.empty_cache()

# Load the original base model to CPU first to avoid accelerate's device_map logic issues
# We remove device_map="auto" to prevent the TypeError.
base_model = AutoModelForCausalLM.from_pretrained(
    ORIGINAL_MODEL_NAME
)

# Load the instruction-tuned adapters onto the base model
instruction_tuned_model_for_merge = PeftModel.from_pretrained(
    base_model,
    INSTRUCTION_ADAPTERS_PATH
)

# Merge the LoRA adapters with the base model and unload them
merged_instruction_model = instruction_tuned_model_for_merge.merge_and_unload()

# Load the tokenizer (the one used during training)
tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Create the output directory if it doesn't exist
os.makedirs(OUTPUT_MERGED_MODEL_DIR, exist_ok=True)

# Save the merged model and tokenizer
merged_instruction_model.save_pretrained(OUTPUT_MERGED_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MERGED_MODEL_DIR)

print(f"Merged instruction-tuned model and tokenizer saved to {OUTPUT_MERGED_MODEL_DIR}")

# Cleanup to free memory
del base_model
del instruction_tuned_model_for_merge
del merged_instruction_model
del tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged instruction-tuned model and tokenizer saved to /content/merged_tinyllama_instruction


In [ ]:
import shutil
from google.colab import files

# Zip the directory containing the saved merged model and tokenizer
shutil.make_archive("merged_tinyllama_instruction", "zip", "/content/merged_tinyllama_instruction")

# Download the zipped file
files.download("merged_tinyllama_instruction.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-lora.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()